# STEP 3: LangChain Tool Calling 기초

이 노트북에서는 기업 분석에 필요한 계산 Tool 하나로 Tool Calling의 전체 흐름을 확인합니다.

`Tool 정의 → Tool schema 확인 → 모델의 Tool 호출 요청 → Python 함수 실행 → 결과를 모델에 전달 → 최종 답변`

이번 단계에서는 OpenDART API를 Tool로 연결하지 않습니다. LangGraph, LlamaIndex, Agent도 사용하지 않습니다.

## 0. Tool Calling이란?

LLM이 Python 함수를 직접 실행하는 것은 아닙니다. 먼저 어떤 함수를 어떤 인자로 실행해야 하는지 **요청**합니다. 그 요청을 받은 Python 코드가 실제 함수를 실행하고, 결과를 다시 모델에 전달합니다.

이번 예제의 Tool은 영업이익률을 계산합니다. 실제 재무데이터는 아직 사용하지 않고, 숫자를 질문에 직접 넣습니다.

## 1. 필요한 클래스 불러오기

- `@tool`: 일반 Python 함수를 Tool로 바꿉니다. 함수 이름, docstring, 타입 힌트가 Tool schema가 됩니다.
- `ToolMessage`: 실제 Tool 실행 결과를 모델에 전달하는 메시지입니다.
- `ChatOllama`: 로컬 Qwen 모델을 연결합니다.

In [ ]:
import json
from urllib.request import urlopen

from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_ollama import ChatOllama

## 2. 로컬 모델 연결 확인

`localhost`는 Jupyter kernel이 실행되는 컴퓨터를 뜻합니다. 모델을 호출하기 전에 Ollama 서버와 모델이 준비되었는지 확인합니다.

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3.6:27b"

with urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=5) as response:
    available_models = [model["name"] for model in json.load(response)["models"]]

if MODEL_NAME not in available_models:
    raise ValueError(f"{MODEL_NAME} 모델을 Ollama에서 찾지 못했습니다: {available_models}")

print(f"Ollama 연결 성공: {MODEL_NAME}")

## 3. Tool로 만들 Python 함수 정의

영업이익률은 `영업이익 ÷ 매출액 × 100`으로 계산합니다. 이 함수에는 별도 모델 판단이 없고, 입력이 같으면 항상 같은 결과를 반환합니다. 이런 결정적 계산을 LLM에게 맡기지 않고 Tool로 분리합니다.

In [ ]:
@tool
def calculate_operating_margin(revenue: float, operating_income: float) -> str:
    """매출액과 영업이익으로 영업이익률(%)을 계산한다. 두 값은 같은 단위를 사용해야 한다."""
    if revenue <= 0:
        raise ValueError("매출액은 0보다 커야 합니다.")

    margin = operating_income / revenue * 100
    return f"영업이익률: {margin:.2f}%"

## 4. Tool schema 확인

LLM은 Python 함수를 읽지 못합니다. LangChain은 함수 이름, 설명, 타입 힌트를 JSON Schema로 바꿔 모델에 전달합니다. 아래 출력에서 Tool 이름과 필요한 인자를 확인하세요.

In [ ]:
calculate_operating_margin.args_schema.model_json_schema()

## 5. 모델에 Tool을 사용할 수 있다고 알리기

`bind_tools()`는 Tool schema를 모델 호출에 포함합니다. 이 단계만으로는 함수가 실행되지 않습니다.

현재 `ChatOllama`는 `tool_choice` 강제 옵션을 지원하지 않습니다. 따라서 system 메시지에서 계산에는 Tool을 쓰도록 지시하고, 다음 셀에서 실제 Tool 요청이 생성됐는지 확인합니다. 나중에 실제 OpenDART Tool을 비교할 때는 이 지시문과 Tool 설명이 Tool 선택 정확도에 영향을 주는 요소가 됩니다.

In [ ]:
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

llm_with_tools = llm.bind_tools([calculate_operating_margin])

## 6. 모델에 계산 요청하기

첫 모델 호출의 목표는 최종 답변이 아닙니다. `tool_calls`에 함수 이름과 모델이 추출한 인자가 들어 있는지 확인하는 것입니다.

In [ ]:
question = "매출액이 1000억 원이고 영업이익이 120억 원일 때 영업이익률을 계산해줘."
system_message = SystemMessage(content="계산이 필요하면 반드시 제공된 Tool을 사용한다.")
initial_messages = [system_message, HumanMessage(content=question)]

tool_request = llm_with_tools.invoke(initial_messages)

tool_request.tool_calls

## 7. 모델이 요청한 Tool과 인자 확인

여기서 보이는 것은 함수 실행 결과가 아니라 **실행 요청서**입니다. 보통 `name`, `args`, `id`를 확인합니다. `id`는 나중에 Tool 결과를 원래 요청과 연결할 때 사용합니다.

In [ ]:
if not tool_request.tool_calls:
    raise RuntimeError("모델이 Tool 호출을 요청하지 않았습니다. tool_request 내용을 확인해 주세요.")

tool_call = tool_request.tool_calls[0]
print("Tool 이름:", tool_call["name"])
print("Tool 인자:", tool_call["args"])
print("Tool 호출 ID:", tool_call["id"])

## 8. Python에서 Tool을 실제로 실행하기

지금부터가 모델 밖에서 실행되는 부분입니다. 사람이 직접 Python 함수를 호출하는 것과 같습니다. Agent나 LangGraph는 나중에 이 실행·반복 과정을 자동화하는 도구입니다.

In [ ]:
if tool_call["name"] != calculate_operating_margin.name:
    raise ValueError(f"예상하지 않은 Tool 요청입니다: {tool_call['name']}")

tool_result = calculate_operating_margin.invoke(tool_call["args"])
print(tool_result)

## 9. Tool 결과를 모델에 돌려주기

모델은 방금 Python 함수가 실행된 사실을 자동으로 알지 못합니다. 첫 호출의 원래 메시지, 모델의 Tool 요청, Tool 결과를 같은 순서로 다시 전달해야 결과를 근거로 최종 답변을 만들 수 있습니다.

In [ ]:
messages = [
    *initial_messages,
    tool_request,
    ToolMessage(content=tool_result, tool_call_id=tool_call["id"]),
]

final_response = llm.invoke(messages)
print(final_response.content)

## 이번 단계에서 반드시 이해할 개념

1. Tool은 `함수 + 설명 + 입력 schema`입니다.
2. `bind_tools()`는 Tool을 실행하지 않고, 모델이 호출할 수 있도록 schema를 전달합니다.
3. 모델의 `tool_calls`는 Python 함수 호출 요청이며, 실제 실행은 애플리케이션 코드가 담당합니다.
4. Tool 결과는 `ToolMessage`로 모델에 다시 전달해야 최종 답변이 그 결과를 사용할 수 있습니다.
5. 지금은 한 번의 Tool 호출을 사람이 순서대로 제어합니다. 이 반복 제어를 자동화하는 것은 아직 다음 단계의 일입니다.